In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("harshshinde8/movies-csv")

print("Path to dataset files:", path)

100%|██████████| 5.13M/5.13M [00:00<00:00, 5.64MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/harshshinde8/movies-csv/versions/1


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import os

# Assuming 'path' variable from the previous cell holds the directory of the downloaded dataset
# Find the CSV file within the downloaded directory
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded dataset directory.")

# Take the first CSV file found
csv_file_path = os.path.join(path, csv_files[0])

df = pd.read_csv(csv_file_path)


# Fill missing values
df["genres"] = df["genres"].fillna("")
df["overview"] = df["overview"].fillna("")

# Combine features
df["content"] = df["genres"] + " " + df["overview"]

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df["content"])


cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


indices = pd.Series(df.index, index=df["title"]).drop_duplicates()


def recommend(title, top_n=5):
    if title not in indices:
        return "Movie not found!"

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [i[0] for i in sim_scores]

    return df["title"].iloc[movie_indices]


print(recommend("XO Kitty"))

Movie not found!
